# 05 · ML demand forecaster: trained, compared with the baseline, registered in Unity Catalog (executed)

`src/jobs/train_demand_forecast.py` trains a gradient-boosted (Poisson) model on store × drink × day demand
from `gold_product_demand`, holds out the last 14 days, and compares it with the heuristic the preparation
plan used before (same-weekday average). It only gets the `@champion` alias if it wins.
The daily job retrains and scores the next 7 days into `ml_demand_forecast` → Lakebase `lb_demand_forecast`.

In [1]:
import warnings; warnings.filterwarnings('ignore')
from proof_helpers import *
print('catalog.schema:', FQ)

catalog.schema: fevm_cme_conde_catalog.kaapi_bricks


In [2]:
import mlflow
from mlflow import MlflowClient
mlflow.set_tracking_uri("databricks"); mlflow.set_registry_uri("databricks-uc")
c = MlflowClient()
name = f"{FQ}.demand_forecaster"
champ = c.get_model_version_by_alias(name, "champion")
print("UC model:", name, "| @champion = version", champ.version, "| status", champ.status)
run = c.get_run(champ.run_id)
m = run.data.metrics
print("training run:", champ.run_id, "| holdout days:", run.data.params["holdout_days"],
      "| train rows:", run.data.params["train_rows"], "| holdout rows:", run.data.params["holdout_rows"])
print("features:", run.data.params["features"])
print()
print(pd.DataFrame([{"method": k, "MAE": round(m[k + "_mae"], 3), "WAPE": round(m[k + "_wape"], 3)}
                    for k in ("model", "baseline_same_weekday", "baseline_28d_mean")]).to_string(index=False))
print(f"\nMAE improvement vs same-weekday baseline: {m['mae_improvement_vs_same_weekday_pct']:.1f}%")

UC model: fevm_cme_conde_catalog.kaapi_bricks.demand_forecaster | @champion = version 2 | status READY


training run: 43a399e3f4d14072bdc381d122a2827d | holdout days: 14 | train rows: 77700 | holdout rows: 14504
features: store_idx,product_idx,dow,is_weekend,is_holiday,lag_7,mean_28,same_dow_mean_4

               method   MAE  WAPE
                model 1.423 0.473
baseline_same_weekday 1.569 0.521
    baseline_28d_mean 1.489 0.495

MAE improvement vs same-weekday baseline: 9.3%


In [3]:
print("All registered versions:")
for v in c.search_model_versions(f"name = '{name}'"):
    print(f"  v{v.version}  aliases={c.get_model_version(name, v.version).aliases}  run={v.run_id}")
print("\nScored forecast (next 7 days):")
print(df(f"""SELECT MIN(forecast_date) AS first_day, MAX(forecast_date) AS last_day, COUNT(*) AS rows,
              COUNT(DISTINCT store_id) AS stores, COUNT(DISTINCT product_id) AS drinks, MAX(model_version) AS model_version,
              MAX(scored_at) AS scored_at FROM {FQ}.ml_demand_forecast""").to_string(index=False))
print("\nKoramangala, today:")
print(df(f"""SELECT product_name, forecast_units, baseline_same_weekday, is_weekend, is_holiday
              FROM {FQ}.ml_demand_forecast WHERE store_id = 'STR-001' AND forecast_date = current_date()
              ORDER BY forecast_units DESC LIMIT 8""").to_string(index=False))

All registered versions:


  v2  aliases=['champion']  run=43a399e3f4d14072bdc381d122a2827d


  v1  aliases=[]  run=c7fd690743e24549a342bb1cd959cb6d

Scored forecast (next 7 days):


 first_day   last_day rows stores drinks model_version                scored_at
2026-10-23 2026-10-29 7252     37     28             2 2026-10-23T18:06:51.000Z

Koramangala, today:


            product_name forecast_units baseline_same_weekday is_weekend is_holiday
   Classic Filter Coffee          14.08                 12.25          0          0
        Strong Decoction          10.63                 11.25          0          0
             Masala Chai           9.76                  7.25          0          0
           Degree Coffee           8.33                   7.0          0          0
   Bella Kaapi (Jaggery)            6.6                   8.0          0          0
             Cold Coffee           5.26                   5.5          0          0
    Mysore Filter Coffee           4.94                  3.25          0          0
Kumbakonam Degree Coffee           4.49                  4.75          0          0
